# Fine-Tuning

## Задача

Выполнить Fine-Tuning по стратегии прогрессии с фиксацией ключевых паараметров для решения задачи классификации обращений по проблемам в городе, представляющих собой текстовые сообщения

### Источник
Задача выбрана на основе вариантов из https://huggingface.co/datasets во вкладке Tasks/Natural Language Processing/Text Classification - Adilbai/kz-gov-complaints-data-kz-ru

### Ограничения
Обучение ограничено возможностями ноутбука MSI Sword 15 A12UE-487XRU, оснащённом GPU NVIDIA GeForce RTX 3060 с 6 ГБ видеопамяти

## Основные зависимости
1. transformers, peft для обучения
2. sklearn.metrics и numpy - в рамках подсчета метрик качества
3. datasets - для загрузки и работы с набором данных для обучения и оценки
4. os - для возможности работы с путями и директориями
5. matplotlib - дя графической интепретации результатов

In [ ]:
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    BitsAndBytesConfig,
    TrainingArguments, 
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import f1_score, accuracy_score
from datasets import load_dataset
import numpy as np
import os
import matplotlib.pyplot as plt

## Гиперпараметры обучения
- Ранг r для LoRA
- Мастабирующий коэффициент a (альфа) - вклад полученной матрицы
- Скорость обучения
- Число эпох
- Разбер батча

In [ ]:
R = 32
A = 128
LR = 1e-4
EPOCH = 3
BATCH = 2

## Параметры формирования набора данных
- split - доля данных для eval
- seed - зерно генерации (чтобы разделение на train/val выполнялось каждый раз по одному сценриаю)

In [ ]:
SPLIT = 0.10
SEED = 44

## Сохранение отчетов об обучении

In [ ]:
REP_DIR = './reports'
FIG_DIR = './figs'

if not os.path.exists(FIG_DIR):
    os.mkdir(FIG_DIR)

if not os.path.exists(REP_DIR):
    os.mkdir(REP_DIR)
    with open(os.path.join(REP_DIR, "match"), "a") as f:
        f.write(f"Try\tModel\tQuant\tBatch\tEpochs\ttLearningRate\ta\tr\n")

TRY = len(os.listdir(REP_DIR)) - 1
REP_NAME = f'try-{TRY}.txt'

## Модели для прогрессивной стратегии

### Основные модели
1. Малая модель для проверки работы пайплайна: Bingsu/llama-190m-arch
2. Модель "среднего" размера для тестирования набора данных: Qwen/Qwen2.5-0.5B-Instruct
3. Модель "крупная" для определения верхнего предела: Qwen/Qwen2.5-3B-Instruct
### Доп. модели для промежуточных оценок
1. google/gemma-3-270m
2. google/gemma-3-1b-it
3. Qwen/Qwen2.5-1.5B-Instruct

In [ ]:
# MODEL_NAME, QUANT = "Bingsu/llama-190m-arch", "no" # 2.5 min
# MODEL_NAME, QUANT = "Qwen/Qwen2.5-0.5B-Instruct", "no" # 4 min
# MODEL_NAME, QUANT = "Qwen/Qwen2.5-3B-Instruct", "4bit" # 60 min


MODEL_NAME, QUANT = "google/gemma-3-270m", "no" # 3.5 min
# MODEL_NAME, QUANT = "google/gemma-3-1b-it", "no" # 8 min
# MODEL_NAME, QUANT = "Qwen/Qwen2.5-1.5B-Instruct", "no" # 10 min

## Набор данных и его подготовка

### Описание
Набор данных Adilbai/kz-gov-complaints-data-kz-ru содержит 1200 созданных Gemini 2.5 записей обращений горожан по городским проблемам на русском и казахском языках. Записи разделены на 10 категорий. Каждая категория занимает 10% +- 0.3% записией, что говорит о сбалансированности классов.

### Стркутура
Данные имеют табличный вид с аттрибутами: 
- id - уникальный идентификатор обращения int64, 
- text_kz - текст обращения на казахском string, 
- text_ru - текст обращения на русском string, 
- category - класс обращения string, 
- urgency - степень срочности string, 
- region - регион происшествия string, 
- status - статус обращения string, 
- reply_text - ответ оператора string, 
- duplicate - повторное ли обращения bool,

### План преобразования
Для задачи классификации выделим из данного набора обращения на русском языке и катеории. Категории преобразуем в численный вид, путем сопоставления класс - номер класса. Далее полученную таблицу разделем в заданном соотношении на две части - обучение train и оценка val

### Примеры пар для классификаии

|На вход - текст|Имя класса |На выходе - номер класса (int) |
|-------|--------|------|
|В городе Шымкент по проспекту Тауке хана мусорные контейнеры постоянно переполнены. Их не вывозят вовремя, распространяется неприятный запах. Помогите решить проблему.|ЖКХ|0|
|В Туркестанской области очень низкое качество интернета. Часто происходят обрывы, скорость низкая. Прошу повлиять на улучшение работы провайдеров.|интернет|2|
|К участковому врачу нужно записываться за месяц. А как попасть на прием, если случай срочный?|медицина|4|

### Преобразования
1. Для проведения классификации обращений по категориям выберем из представленного набора столбцы 'text_ru' и 'category'

In [ ]:
data = load_dataset("Adilbai/kz-gov-complaints-data-kz-ru")
core = data['train']
dataset = core.select_columns(["text_ru", "category"])

2. Извлечем категории и сопоставим их с числовым эквивалентом

In [ ]:
unique_categories = sorted(set(dataset['category']))
print(unique_categories)
cat_to_id = {cat: i for i, cat in enumerate(unique_categories)}
dataset = dataset.map(lambda x: {"labels": cat_to_id[x["category"]]})
dataset = dataset.remove_columns(["category"])

3. Разобъем данные на обучащие и валидационные в ранее определенном соотношении

In [ ]:
dataset = dataset.train_test_split(test_size=SPLIT, seed=SEED)

4. Выполним токенизацию

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
def tokenize(examples):
    return tokenizer(
        examples["text_ru"], 
        truncation=True, 
        padding="max_length", 
        max_length=324
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

## Загрузка модели
Загрузим выбранную модель и, при необходимости, преобразуем ее (16bit -> 4/8bit)

In [ ]:
model = None

if QUANT != "no":
    bnb_config = None
    
    if QUANT == "4bit":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
        )
    else:
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
        )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(unique_categories),
        quantization_config=bnb_config,
        device_map="cuda",
        trust_remote_code=True,
    )

else:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(unique_categories),
        device_map="cuda",
        trust_remote_code=True,
    )
    
model.config.pad_token_id = tokenizer.pad_token_id

## Настройки LoRA/QLoRA
Определим параметры LoRA/QLoRA, указав:
- К каким матрицам применияется (Q, K, V и Output)
- Ранг r
- Масштабирующий коэффициент a

In [ ]:
lora_config = LoraConfig(
    r=R,
    lora_alpha=A,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)

## Метрики оценки качества классификации
- Поскольку классы сбалансированы можно применить **Accuracy**
- В дополнение к ней также рассмотрим **Macro-F1**, которая дает учитывать точность и полноту классификации по каждому классу. 
Ожидается, что значения Accuracy и Macro-F1 будут близки, однако F1-мера используется для дополнительной проверки устойчивости модели.

In [ ]:
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=-1)
    return {
        "accuray": accuracy_score(eval_pred.label_ids, predictions),
        "f1": f1_score(eval_pred.label_ids, predictions, average='macro')
    }

## Подготовка процесса обучения
Нужно определенить настройки обучения:
- Выходная директория с лучшими адаптерами
- Размер батча
- Число эпох обучения
- Скорость обучения
- Оптимизатор
- Частота оценки и сохранения, логирование
- Метрика определения лучшей модели

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./results-{TRY}",
    save_total_limit=3,
    load_best_model_at_end=True,
    
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=4,

    num_train_epochs=EPOCH,
    learning_rate=LR,
    
    optim="paged_adamw_8bit",

    eval_strategy="steps",
    eval_steps=15,

    save_strategy="steps",
    save_steps=15,
    
    metric_for_best_model="f1",
    logging_steps=15,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

## Запуск обучения

In [ ]:
trainer.train()

## Сохранение истории обучения

In [ ]:
train_losses = []
eval_losses = []
eval_acc = []
eval_f1 = []
steps = []

for log in trainer.state.log_history:
    if 'loss' in log:
        steps.append(log['step'])
        train_losses.append(log['loss'])
    if 'eval_loss' in log:
        eval_losses.append(log['eval_loss'])
        eval_f1.append(log['eval_f1'])
        eval_acc.append(log['eval_accuray'])

Графическая интерпретация истории обучения

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

ax1.plot(steps, train_losses, 'o-', label='Train Loss', linewidth=2, markersize=4, color='blue')
ax1.plot(steps, eval_losses, 's-', label='Eval Loss', linewidth=2, markersize=4, color='red')
ax1.set_xlabel('Step', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Train and Eval Loss', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(steps, eval_acc, 's-', label='Accuracy', linewidth=2, markersize=4, color='green')
ax2.plot(steps, eval_f1, 's-', label='F1 Score', linewidth=2, markersize=4, color='purple')
ax2.set_xlabel('Step', fontsize=12)
ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Accuracy and F1 Score', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)


plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'fig-{TRY}.png'), dpi=150, bbox_inches='tight')
plt.show()

Сохранение в файл для построения будущих отчетов и документирования результатов

In [ ]:
with open(os.path.join(REP_DIR, "match"), "a") as f:
    f.write(f"{TRY}\t{MODEL_NAME}\t{QUANT}\t{BATCH}\t{EPOCH}\t{LR}\t{A}\t{R}\n")

with open(os.path.join(REP_DIR, REP_NAME), "w") as f:
    f.write(f"step\tloss\teval_loss\tF1\tAcc\n")

    for step, loss, eval_loss, f1, acc in zip(steps, train_losses, eval_losses, eval_f1, eval_acc):
        f.write(f"{step}\t{loss}\t{eval_loss}\t{f1}\t{acc}\n")

## Собранные результаты
1. Для оценкии затрат выбран - Размер итоговой модели - определяет объём занимаемой памяти, а также может указывать на требования к аппаратному обеспечению, если появится необходимость в размещении такой модели
2. Для оценки качества можем выбрать как F1-Score, так и Accuracy (практически не отличались во всех опытах)

Обработав все полученные файлы, получили:

### Обзор влияния гиперпараметров a,r:

|Модель|a|r|a/r|Лучшая Macro-F1|Лучшый Шаг (из 405)|
|------|-|-|---|-----------------|-----|
|Bingsu/llama-190m-arch|32|8|4|0.6834|300|
|Qwen/Qwen2.5-0.5B-Instruct|32|8|4|0.9690|315|
|Qwen/Qwen2.5-3B-Instruct-4bit|32|8|4|0.9782|270|
|Bingsu/llama-190m-arch|8|2|4|0.5778|255|
|Qwen/Qwen2.5-0.5B-Instruct|8|2|4|0.9450|390|
|Qwen/Qwen2.5-3B-Instruct-4bit|8|2|4|0.9740|360|
|Bingsu/llama-190m-arch|128|32|4|0.7903|405|
|Qwen/Qwen2.5-0.5B-Instruct|128|32|4|0.9643|315|
|Qwen/Qwen2.5-3B-Instruct-4bit|128|32|4|0.9713|255|
|Bingsu/llama-190m-arch|8|8|1|0.5827|255|
|Qwen/Qwen2.5-0.5B-Instruct|8|8|1|0.9282|255|
|Qwen/Qwen2.5-3B-Instruct-4bit|8|8|1|0.9636|300|
|Bingsu/llama-190m-arch|2|8|0.25|0.5526|300|
|Qwen/Qwen2.5-0.5B-Instruct|2|1|0.25|0.9103|390|
|Qwen/Qwen2.5-3B-Instruct-4bit|2|8|0.25|0.9038|390|

Исходя из сведений в таблице:
1. Уменьшение отношения a/r ослабляет влияние прибавляемых весов, что приводит к недостаточному качеству
2. Уменьшение параметра r ограничивает размерность адаптационного пространства, что снижает гибкость модели и ведет к падению метрик качества или не достижению желаемого качества за выбранные эпохи (недообучение)
3. Оптимальное соотношение для выбранной задачи составляет a/r = 4. Для средней и крупной модели при r=8, a=32. Для малой модели при r=32, a=128

Исходя из полученных выводов, выберем пару a=32 и r=8 как единую ось для итогового сравнения.

### Дополнительные эксперементы при a=32 и r=8 дали следующие результаты:

|Модель|a|r|a/r|Лучшая Macro-F1|Лучшый Шаг (из 405)|
|------|-|-|---|---------------|------------|
|google/gemma-3-270m|32|8|4|0.9660|405|
|google/gemma-3-1b-it|32|8|4|0.9782|225|
|Qwen/Qwen2.5-1.5B-Instruct|32|8|4|0.9786|240|

## Формирование итоговых результатов

Сформируем итоговую сравнительную таблицу на выбранной оси (a=32 и r=8):

|Модель|Качество (Macro-F1)|Размер (GB)|
|------|-------------------|-----------|
|Bingsu/llama-190m-arch|0.6834|~0.4|
|google/gemma-3-270m|0.9660|~0.6|
|Qwen/Qwen2.5-0.5B-Instruct|0.9690|~1.0|
|Qwen/Qwen2.5-3B-Instruct-4bit|0.9782|~1.5|
|google/gemma-3-1b-it|0.9782|~2.0|
|Qwen/Qwen2.5-1.5B-Instruct|0.9786|~3.0|

Отобразим полученные результаты на диаграмме рассеяния

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names = [
    "Bingsu/llama-190m-arch", "google/gemma-3-270m", "Qwen/Qwen2.5-0.5B-Instruct", 
    "google/gemma-3-1b-it", "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct-4bit"
]
gb = [0.4, 0.6, 1.0, 2.0, 3.0, 1.5]
f1s = [0.6834, 0.9660, 0.9690, 0.9782, 0.9786, 0.9782]

plt.figure(figsize=(10, 6))
plt.scatter(gb, f1s, s=100, c='steelblue', alpha=0.7, edgecolors='darkblue', linewidth=2, zorder=5)

short_names = [n.split('/')[-1].replace('-Instruct', '') for n in names]
for i, name in enumerate(short_names):
    plt.annotate(name, (gb[i], f1s[i]), 
                xytext=(-25, -15), textcoords='offset points',
                fontsize=9, alpha=0.8)

plt.xlabel('Размер модели (ГБ)', fontsize=12, fontweight='bold')
plt.ylabel('Качество (Macro-F1)', fontsize=12, fontweight='bold')
plt.title('Диаграмма рассеяния', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.xlim(0, 3.2)
plt.ylim(0.65, 0.99)

plt.tight_layout()
plt.show()

## Выбор
Исходя из представленных диаграмм, оптимальным выбором можно считать модель **google/gemma-3-270m**, которая при 270m параметров (0.5-0.6 GB), достигла 0.966 F1-Score (на 1.2% меньше, чем у лучших по качеству) на задаче классификации Adilbai/kz-gov-complaints-data-kz-ru при обучении с использованием LoRA (r=8, a=32), которое заняло 2.5 минуты